In [ ]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import paste as pst
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.insert(0, './')
from metrics import iLISI

In [ ]:
def stack_obs(_dir):
    fds = os.listdir(_dir)
    _obs = []
    for fd in fds:
        if fd.startswith("R"):
            continue
        fns = os.listdir(join(_dir, fd))
        adx = sc.read_h5ad(join(_dir, fd, fns[0]))
        _obs.append(adx.obs)
    df = pd.concat(_obs)
    return df

In [ ]:
def calc_iLISI(data_dir, emb_dir, ds, batch_key):
    df_meta = stack_obs(join(data_dir, ds))
    
    df_res = {}
    for meth in target_methods:
        emb_file = join(emb_dir, meth, ds, 'df_emb.csv')
        if not os.path.exists(emb_file):
            print(f"[Skip] {emb_file} not found.")
            continue

        df_emb = pd.read_csv(emb_file, index_col=0)

        common_idx = df_meta.index.intersection(df_emb.index)
        if len(common_idx) == 0:
            print(f"[Warning] No overlapping cells between meta and cluster for {meth}")
            continue

        df_meta_aligned = df_meta.loc[common_idx].copy()
        df_emb_aligned = df_emb.loc[common_idx].copy()

        adx = sc.AnnData(np.zeros((len(common_idx), 2)))
        adx.obs = df_meta_aligned.copy()
        adx.obsm['emb'] = df_emb_aligned.values

        df_res[meth] = iLISI(adx, batch_key, 'emb')

    df_res = pd.DataFrame.from_dict(df_res, orient='index', columns=['iLISI'])
    return df_res


In [ ]:
target_methods = [
    'CLUE',
    'StabMap',
    'SpaMosaic(non-spatial)',
    'UINMF',
    'scMoMaT',
    'SpaMosaic',
    'MIDAS',
    'Cobolt',
    'MultiVI'
]

data_dir = '../../data/processed/'
emb_dir = '../../results/embeddings/'
score_dir = '../../results/scores/'

### DBiT-RnaAtac-Atac

In [ ]:
ds = 'DBiT-RnaAtac-Atac'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Embryo

In [ ]:
ds = 'Embryo'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Lymph-S1-S2-S3

In [ ]:
ds = 'Lymph-S1-S2-S3'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### MB-5M

In [ ]:
ds = 'MB-5M'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### MB-H3K4me3

In [ ]:
ds = 'MB-H3K4me3'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Misar-E13-E15-E18

In [ ]:
ds = 'Misar-E13-E15-E18'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'Sample')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Misar-Stereo

In [ ]:
ds = 'Misar-Stereo'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'Sample')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Mux-VisiumHD-HE

In [ ]:
ds = 'Mux-VisiumHD-HE'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'Slice')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Tonsil-S1-S2-S3

In [ ]:
ds = 'Tonsil-S1-S2-S3'

dfl = calc_iLISI(data_dir, emb_dir, ds, 'src')
dfl.to_csv(join(score_dir, ds, 'df_iLISI.csv'))

### Simulations

In [ ]:
def stack_obs2(data_dir):
    ad1 = sc.read_h5ad(join(data_dir, 'ad_rna1.h5ad'))
    ad2 = sc.read_h5ad(join(data_dir, 'ad_rna2.h5ad'))
    ad3 = sc.read_h5ad(join(data_dir, 'ad_adt2.h5ad'))
    
    ad1.obs['src'] = 's1'
    ad2.obs['src'] = 's2'
    ad3.obs['src'] = 's3'
    dft = pd.concat([ad1.obs, ad2.obs, ad3.obs])
    return dft

In [ ]:
ds = 'Simulations'
fds = os.listdir(join(data_dir, ds))

for fd in fds:
    for seed in ['1', '2', '3']:
        df_meta = stack_obs2(join(data_dir, ds, fd, seed))

        df_res = {}
        for meth in target_methods:
            emb_file = join(emb_dir, meth, ds, fd, seed, 'df_emb.csv')
            if not os.path.exists(emb_file):
                print(f"[Skip] {emb_file} not found.")
                continue

            df_emb = pd.read_csv(emb_file, index_col=0)

            common_idx = df_meta.index.intersection(df_emb.index)
            if len(common_idx) == 0:
                print(f"[Warning] No overlapping cells between meta and cluster for {meth}")
                continue

            df_meta_aligned = df_meta.loc[common_idx].copy()
            df_emb_aligned = df_emb.loc[common_idx].copy()

            adx = sc.AnnData(np.zeros((len(common_idx), 2)))
            adx.obs = df_meta_aligned.copy()
            adx.obsm['emb'] = df_emb_aligned.values

            df_res[meth] = iLISI(adx, 'src', 'emb')

        df_res = pd.DataFrame.from_dict(df_res, orient='index', columns=['iLISI'])
        
        os.makedirs(join(score_dir, ds, fd, seed), exist_ok=True)
        df_res.to_csv(join(score_dir, ds, fd, seed, 'df_iLISI.csv'))

In [ ]:
df_res